# ABC CAD Dataset - CSTBIR Evaluation

This notebook evaluates the trained CSTBIR model on the ABC CAD dataset.

**Setup:**
- Query set: Validation split (10% of dataset, ~20k samples)
- Gallery set: Training split (90% of dataset, ~180k samples)

**Tasks:**
1. SBIR (Sketch-Based Image Retrieval): sketch query → image gallery
2. TBIR (Text-Based Image Retrieval): text query → image gallery
3. STBIR (Sketch+Text-Based Image Retrieval): combined query → image gallery

**Metrics:**
- mAP@k (Mean Average Precision at k)
- Recall@k (R@k)
- k ∈ {1, 5, 10, 50}

In [1]:
# Cell 1: Imports and Setup
import os
import torch
import numpy as np
import torch.nn.functional as F
from tqdm import tqdm
import matplotlib.pyplot as plt
from PIL import Image
from src import clip
from dataloader import get_dataloader
from utils import load_config
import pandas as pd

# Load configuration
config = load_config('config.yml')

# Set up device
device = "cuda:0" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Load CLIP model
model, preprocess = clip.load(config['model']['model_name'], device=device, jit=False)

# Load the trained model
checkpoint_path = 'data/abc_finetuned/model_epoch_15.pt'  # Change to desired epoch
print(f"Loading checkpoint: {checkpoint_path}")
model.load_state_dict(torch.load(checkpoint_path, map_location=device))
model.eval()

print("Model loaded successfully!")

Using device: cuda:0
Loading checkpoint: data/abc_finetuned/model_epoch_15.pt
Model loaded successfully!


In [2]:
# Cell 2: Generate Embeddings for Validation Set (Queries)

# Load CSV once to create index-to-text mapping
print("Loading CSV for text mapping...")
df_full = pd.read_csv(config['data']['csv_path'])
index_to_text = dict(zip(df_full.index, df_full['text']))
print(f"Loaded {len(index_to_text)} text entries")

def generate_embeddings(data_loader, device, index_to_text_map):
    """Generate embeddings for all samples in the dataloader"""
    image_embeddings = []
    sketch_embeddings = []
    text_embeddings = []
    image_paths = []
    sketch_paths = []
    text_prompts = []
    original_indices = []
    
    with torch.no_grad():
        for batch in tqdm(data_loader, desc="Generating embeddings"):
            images, sketches, texts, img_paths, sk_paths, indices = batch
            
            images = images.to(device)
            sketches = sketches.to(device)
            texts = texts.to(device)
            
            # Encode
            image_feat = model.encode_image(images)
            sketch_feat = model.encode_image(sketches)
            text_feat = model.encode_text(texts)
            
            # Normalize
            image_feat = F.normalize(image_feat, p=2, dim=1)
            sketch_feat = F.normalize(sketch_feat, p=2, dim=1)
            text_feat = F.normalize(text_feat, p=2, dim=1)
            
            image_embeddings.append(image_feat.cpu())
            sketch_embeddings.append(sketch_feat.cpu())
            text_embeddings.append(text_feat.cpu())
            
            image_paths.extend(img_paths)
            sketch_paths.extend(sk_paths)
            # Use mapping from original CSV indices to texts
            text_prompts.extend([str(index_to_text_map[i.item()]) for i in indices])
            original_indices.extend(indices.tolist())
    
    return (
        torch.cat(image_embeddings),
        torch.cat(sketch_embeddings),
        torch.cat(text_embeddings),
        image_paths,
        sketch_paths,
        text_prompts,
        original_indices
    )

# Create dataloader for validation set (queries)
print("\n=== Loading Validation Set (Queries) ===")
val_loader = get_dataloader(
    config['data']['csv_path'],
    config['data']['image_path'],
    config['data']['sketch_path'],
    batch_size=32,
    split='val',
    preprocess=preprocess,
    num_workers=4,
    shuffle=False,
    return_paths=True
)

# Generate query embeddings
print(f"Total validation samples: {len(val_loader.dataset)}")
query_data = generate_embeddings(val_loader, device, index_to_text)

# Save query embeddings
torch.save({
    'image_embeddings': query_data[0],
    'sketch_embeddings': query_data[1],
    'text_embeddings': query_data[2],
    'image_paths': query_data[3],
    'sketch_paths': query_data[4],
    'text_prompts': query_data[5],
    'original_indices': query_data[6]
}, 'abc_embeddings_val_queries.pt')

print(f"\n✓ Query embeddings saved: {len(query_data[0])} samples")

Loading CSV for text mapping...
Loaded 201034 text entries

=== Loading Validation Set (Queries) ===
Total validation samples: 20104


Generating embeddings: 100%|██████████| 629/629 [14:50<00:00,  1.42s/it]



✓ Query embeddings saved: 20104 samples


In [3]:
# Cell 3: Generate Embeddings for Training Set (Gallery)

print("\n=== Loading Training Set (Gallery) ===")
train_loader = get_dataloader(
    config['data']['csv_path'],
    config['data']['image_path'],
    config['data']['sketch_path'],
    batch_size=32,
    split='train',
    preprocess=preprocess,
    num_workers=4,
    shuffle=False,
    return_paths=True
)

# Generate gallery embeddings (use the same index_to_text mapping)
print(f"Total training samples: {len(train_loader.dataset)}")
gallery_data = generate_embeddings(train_loader, device, index_to_text)

# Save gallery embeddings
torch.save({
    'image_embeddings': gallery_data[0],
    'sketch_embeddings': gallery_data[1],
    'text_embeddings': gallery_data[2],
    'image_paths': gallery_data[3],
    'sketch_paths': gallery_data[4],
    'text_prompts': gallery_data[5],
    'original_indices': gallery_data[6]
}, 'abc_embeddings_train_gallery.pt')

print(f"\n✓ Gallery embeddings saved: {len(gallery_data[0])} samples")


=== Loading Training Set (Gallery) ===
Total training samples: 180930


Generating embeddings: 100%|██████████| 5655/5655 [08:51<00:00, 10.63it/s]



✓ Gallery embeddings saved: 180930 samples


In [4]:
# Cell 4: Load Embeddings and Prepare for Evaluation

print("Loading embeddings...")

# Load query embeddings (validation set)
query_data = torch.load('abc_embeddings_val_queries.pt')
query_image_emb = query_data['image_embeddings'].float()
query_sketch_emb = query_data['sketch_embeddings'].float()
query_text_emb = query_data['text_embeddings'].float()
query_image_paths = query_data['image_paths']
query_sketch_paths = query_data['sketch_paths']
query_text_prompts = query_data['text_prompts']
query_indices = query_data['original_indices']

# Load gallery embeddings (training set)
gallery_data = torch.load('abc_embeddings_train_gallery.pt')
gallery_image_emb = gallery_data['image_embeddings'].float()
gallery_image_paths = gallery_data['image_paths']
gallery_indices = gallery_data['original_indices']

print(f"Query samples: {len(query_image_emb)}")
print(f"Gallery samples: {len(gallery_image_emb)}")

# Create STBIR query embeddings (average of sketch and text)
query_stbir_emb = (query_sketch_emb + query_text_emb) / 2
query_stbir_emb = F.normalize(query_stbir_emb, p=2, dim=1)

print("\n✓ Embeddings loaded and normalized")

Loading embeddings...
Query samples: 20104
Gallery samples: 180930

✓ Embeddings loaded and normalized


In [5]:
# Cell 5: Compute Rankings (Memory-Efficient)

def compute_top_k_rankings(query_embeddings, gallery_embeddings, k=50, batch_size=64):
    """
    Compute top-k rankings for all queries in a memory-efficient way
    """
    num_queries = query_embeddings.shape[0]
    rankings = torch.zeros(num_queries, k, dtype=torch.long)
    
    for i in tqdm(range(0, num_queries, batch_size), desc=f"Computing top-{k} rankings"):
        end_idx = min(i + batch_size, num_queries)
        query_batch = query_embeddings[i:end_idx].to(device)
        
        # Compute similarities
        similarities = torch.mm(query_batch, gallery_embeddings.to(device).T)
        
        # Get top k rankings
        _, top_indices = torch.topk(similarities, k=k, dim=1)
        rankings[i:end_idx] = top_indices.cpu()
        
        # Clear GPU memory
        torch.cuda.empty_cache()
    
    return rankings

# Compute rankings
k = 50  # Keep top-50 for evaluation
batch_size = 128

print("\n=== Computing Rankings ===")

print("\nComputing SBIR rankings (sketch → image)...")
sbir_rankings = compute_top_k_rankings(query_sketch_emb, gallery_image_emb, k, batch_size)

print("Computing TBIR rankings (text → image)...")
tbir_rankings = compute_top_k_rankings(query_text_emb, gallery_image_emb, k, batch_size)

print("Computing STBIR rankings (sketch+text → image)...")
stbir_rankings = compute_top_k_rankings(query_stbir_emb, gallery_image_emb, k, batch_size)

# Save rankings
torch.save({
    'sbir_rankings': sbir_rankings,
    'tbir_rankings': tbir_rankings,
    'stbir_rankings': stbir_rankings
}, 'abc_rankings.pt')

print("\n✓ Rankings computed and saved")


=== Computing Rankings ===

Computing SBIR rankings (sketch → image)...


Computing top-50 rankings: 100%|██████████| 158/158 [00:07<00:00, 20.86it/s]


Computing TBIR rankings (text → image)...


Computing top-50 rankings: 100%|██████████| 158/158 [00:07<00:00, 20.72it/s]


Computing STBIR rankings (sketch+text → image)...


Computing top-50 rankings: 100%|██████████| 158/158 [00:07<00:00, 21.01it/s]


✓ Rankings computed and saved


In [6]:
# Cell 6: Compute Evaluation Metrics

def compute_mAP_at_k(rankings, k):
    """
    Compute mAP@k using model ID matching from file paths
    For ABC dataset, extract model ID from image filenames
    """
    num_queries = rankings.shape[0]
    ap_sum = 0.0
    
    for i in range(num_queries):
        # Get top k predictions
        top_k_indices = rankings[i, :k]
        
        # Extract query model ID from filename
        # Format: /path/to/100000_iso1.png -> model_id = 100000
        query_path = query_image_paths[i]
        query_model_id = os.path.basename(query_path).split('_')[0]
        
        # Calculate AP for this query
        precision_sum = 0.0
        num_relevant = 0
        
        for j, pred_idx in enumerate(top_k_indices):
            pred_path = gallery_image_paths[pred_idx]
            pred_model_id = os.path.basename(pred_path).split('_')[0]
            
            # Check if same model (relevant)
            if pred_model_id == query_model_id:
                num_relevant += 1
                precision_at_j = num_relevant / (j + 1)
                precision_sum += precision_at_j
        
        # Average precision for this query
        if num_relevant > 0:
            ap = precision_sum / num_relevant
            ap_sum += ap
    
    return ap_sum / num_queries

def compute_recall_at_k(rankings, k):
    """
    Compute Recall@k using model ID matching
    """
    num_queries = rankings.shape[0]
    correct = 0
    
    for i in range(num_queries):
        top_k_indices = rankings[i, :k]
        
        query_path = query_image_paths[i]
        query_model_id = os.path.basename(query_path).split('_')[0]
        
        # Check if any of top-k matches the query model
        for pred_idx in top_k_indices:
            pred_path = gallery_image_paths[pred_idx]
            pred_model_id = os.path.basename(pred_path).split('_')[0]
            
            if pred_model_id == query_model_id:
                correct += 1
                break  # Count once per query
    
    return correct / num_queries

# Evaluate metrics
k_values = [1, 5, 10, 50]

print("\n" + "="*80)
print("EVALUATION RESULTS - ABC CAD Dataset")
print("="*80)

print(f"\nQuery samples: {len(query_image_emb)}")
print(f"Gallery samples: {len(gallery_image_emb)}")

# SBIR Metrics
print("\n" + "-"*80)
print("SBIR (Sketch-Based Image Retrieval)")
print("-"*80)
for k in k_values:
    map_k = compute_mAP_at_k(sbir_rankings, k)
    recall_k = compute_recall_at_k(sbir_rankings, k)
    print(f"mAP@{k:2d}: {map_k:.4f}  |  R@{k:2d}: {recall_k:.4f}")

# TBIR Metrics
print("\n" + "-"*80)
print("TBIR (Text-Based Image Retrieval)")
print("-"*80)
for k in k_values:
    map_k = compute_mAP_at_k(tbir_rankings, k)
    recall_k = compute_recall_at_k(tbir_rankings, k)
    print(f"mAP@{k:2d}: {map_k:.4f}  |  R@{k:2d}: {recall_k:.4f}")

# STBIR Metrics
print("\n" + "-"*80)
print("STBIR (Sketch+Text-Based Image Retrieval)")
print("-"*80)
for k in k_values:
    map_k = compute_mAP_at_k(stbir_rankings, k)
    recall_k = compute_recall_at_k(stbir_rankings, k)
    print(f"mAP@{k:2d}: {map_k:.4f}  |  R@{k:2d}: {recall_k:.4f}")

print("\n" + "="*80)


EVALUATION RESULTS - ABC CAD Dataset

Query samples: 20104
Gallery samples: 180930

--------------------------------------------------------------------------------
SBIR (Sketch-Based Image Retrieval)
--------------------------------------------------------------------------------
mAP@ 1: 0.4642  |  R@ 1: 0.4642
mAP@ 5: 0.5258  |  R@ 5: 0.6289
mAP@10: 0.5325  |  R@10: 0.6795
mAP@50: 0.5372  |  R@50: 0.7715

--------------------------------------------------------------------------------
TBIR (Text-Based Image Retrieval)
--------------------------------------------------------------------------------
mAP@ 1: 0.0480  |  R@ 1: 0.0480
mAP@ 5: 0.0764  |  R@ 5: 0.1315
mAP@10: 0.0845  |  R@10: 0.1933
mAP@50: 0.0946  |  R@50: 0.4119

--------------------------------------------------------------------------------
STBIR (Sketch+Text-Based Image Retrieval)
--------------------------------------------------------------------------------
mAP@ 1: 0.4618  |  R@ 1: 0.4618
mAP@ 5: 0.5381  |  R@ 5: 0.

In [7]:
# Cell 7: Visualization Function

def visualize_results(query_index, rankings, query_type, save_dir='abc_visualizations'):
    """
    Visualize query and top-10 retrieval results
    """
    os.makedirs(save_dir, exist_ok=True)
    
    fig, axes = plt.subplots(2, 6, figsize=(24, 10))
    
    # Query information
    query_model_id = os.path.basename(query_image_paths[query_index]).split('_')[0]
    query_text = query_text_prompts[query_index]
    
    fig.suptitle(
        f"{query_type} Results - Query Model: {query_model_id}\n"
        f"Text: {query_text[:100]}...",
        fontsize=14,
        fontweight='bold'
    )
    
    # Display query
    if query_type in ['SBIR', 'STBIR']:
        query_path = query_sketch_paths[query_index]
        title = "Query\nSketch"
    else:  # TBIR
        query_path = query_image_paths[query_index]
        title = "Query\nImage\n(for reference)"
    
    try:
        query_image = Image.open(query_path).convert('RGB')
        axes[0, 0].imshow(query_image)
    except:
        axes[0, 0].text(0.5, 0.5, "Query not\navailable", ha='center', va='center')
    
    axes[0, 0].set_title(title, fontsize=10, fontweight='bold')
    axes[0, 0].axis('off')
    
    # Display top-10 results
    for i in range(10):
        row = i // 5
        col = (i % 5) + 1
        
        result_idx = rankings[query_index, i].item()
        result_path = gallery_image_paths[result_idx]
        result_model_id = os.path.basename(result_path).split('_')[0]
        
        # Check if correct match
        is_correct = (result_model_id == query_model_id)
        
        try:
            result_image = Image.open(result_path).convert('RGB')
            axes[row, col].imshow(result_image)
        except:
            axes[row, col].text(0.5, 0.5, "Image not\navailable", ha='center', va='center')
        
        title = f"Rank {i+1}\nModel: {result_model_id}"
        color = 'green' if is_correct else 'black'
        weight = 'bold' if is_correct else 'normal'
        
        axes[row, col].set_title(title, fontsize=9, color=color, fontweight=weight)
        axes[row, col].axis('off')
        
        # Add green border for correct matches
        if is_correct:
            for spine in axes[row, col].spines.values():
                spine.set_edgecolor('green')
                spine.set_linewidth(3)
                spine.set_visible(True)
    
    plt.tight_layout()
    
    # Save figure
    save_path = os.path.join(save_dir, f"{query_type}_query_{query_index}_model_{query_model_id}.png")
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    
    print(f"✓ Saved: {save_path}")

print("Visualization function ready!")

Visualization function ready!


In [8]:
# Cell 8: Generate Visualizations for Random Queries

# Number of random queries to visualize
num_visualizations = 10

# Select random query indices
np.random.seed(42)  # For reproducibility
random_query_indices = np.random.choice(len(query_image_emb), num_visualizations, replace=False)

print(f"\nGenerating visualizations for {num_visualizations} random queries...")
print("="*80)

for idx, query_idx in enumerate(random_query_indices, 1):
    print(f"\n[{idx}/{num_visualizations}] Query index: {query_idx}")
    
    # Visualize SBIR
    visualize_results(query_idx, sbir_rankings, "SBIR")
    
    # Visualize TBIR
    visualize_results(query_idx, tbir_rankings, "TBIR")
    
    # Visualize STBIR
    visualize_results(query_idx, stbir_rankings, "STBIR")

print("\n" + "="*80)
print("✓ All visualizations completed!")
print(f"✓ Saved to: abc_visualizations/")


Generating visualizations for 10 random queries...

[1/10] Query index: 18633
✓ Saved: abc_visualizations/SBIR_query_18633_model_190671.png
✓ Saved: abc_visualizations/TBIR_query_18633_model_190671.png
✓ Saved: abc_visualizations/STBIR_query_18633_model_190671.png

[2/10] Query index: 15455
✓ Saved: abc_visualizations/SBIR_query_15455_model_173288.png
✓ Saved: abc_visualizations/TBIR_query_15455_model_173288.png
✓ Saved: abc_visualizations/STBIR_query_15455_model_173288.png

[3/10] Query index: 4445
✓ Saved: abc_visualizations/SBIR_query_4445_model_61573.png
✓ Saved: abc_visualizations/TBIR_query_4445_model_61573.png
✓ Saved: abc_visualizations/STBIR_query_4445_model_61573.png

[4/10] Query index: 11764
✓ Saved: abc_visualizations/SBIR_query_11764_model_30458.png
✓ Saved: abc_visualizations/TBIR_query_11764_model_30458.png
✓ Saved: abc_visualizations/STBIR_query_11764_model_30458.png

[5/10] Query index: 718
✓ Saved: abc_visualizations/SBIR_query_718_model_33104.png
✓ Saved: abc_visua

In [9]:
# Cell 9: Per-View Analysis (Optional)

def analyze_by_view():
    """
    Analyze performance separately for iso1 vs iso2 views
    """
    print("\n" + "="*80)
    print("PER-VIEW ANALYSIS")
    print("="*80)
    
    # Separate queries by view type
    iso1_indices = [i for i, path in enumerate(query_image_paths) if '_iso1' in path]
    iso2_indices = [i for i, path in enumerate(query_image_paths) if '_iso2' in path]
    
    print(f"\nISO1 queries: {len(iso1_indices)}")
    print(f"ISO2 queries: {len(iso2_indices)}")
    
    for view_name, view_indices in [('ISO1', iso1_indices), ('ISO2', iso2_indices)]:
        print(f"\n{'-'*80}")
        print(f"{view_name} View - SBIR Performance")
        print(f"{'-'*80}")
        
        for k in [1, 5, 10]:
            correct = 0
            for i in view_indices:
                query_model_id = os.path.basename(query_image_paths[i]).split('_')[0]
                for pred_idx in sbir_rankings[i, :k]:
                    pred_model_id = os.path.basename(gallery_image_paths[pred_idx]).split('_')[0]
                    if pred_model_id == query_model_id:
                        correct += 1
                        break
            
            recall = correct / len(view_indices)
            print(f"R@{k:2d}: {recall:.4f}")

analyze_by_view()


PER-VIEW ANALYSIS

ISO1 queries: 10102
ISO2 queries: 10002

--------------------------------------------------------------------------------
ISO1 View - SBIR Performance
--------------------------------------------------------------------------------
R@ 1: 0.4627
R@ 5: 0.6270
R@10: 0.6772

--------------------------------------------------------------------------------
ISO2 View - SBIR Performance
--------------------------------------------------------------------------------
R@ 1: 0.4658
R@ 5: 0.6308
R@10: 0.6819


In [10]:
# Cell 10: Save Summary Results

import json

def save_results_summary():
    """
    Save evaluation results to JSON for easy reporting
    """
    results = {
        'dataset': 'ABC CAD Dataset',
        'model': checkpoint_path,
        'num_queries': len(query_image_emb),
        'num_gallery': len(gallery_image_emb),
        'metrics': {}
    }
    
    for task_name, task_rankings in [('SBIR', sbir_rankings), 
                                      ('TBIR', tbir_rankings), 
                                      ('STBIR', stbir_rankings)]:
        results['metrics'][task_name] = {}
        
        for k in [1, 5, 10, 50]:
            map_k = compute_mAP_at_k(task_rankings, k)
            recall_k = compute_recall_at_k(task_rankings, k)
            
            results['metrics'][task_name][f'mAP@{k}'] = float(map_k)
            results['metrics'][task_name][f'R@{k}'] = float(recall_k)
    
    # Save to JSON
    with open('abc_evaluation_results.json', 'w') as f:
        json.dump(results, f, indent=2)
    
    print("\n✓ Results saved to: abc_evaluation_results.json")
    
    # Print formatted table for paper
    print("\n" + "="*80)
    print("RESULTS TABLE (for paper)")
    print("="*80)
    print(f"\n{'Method':<10} {'mAP@1':<10} {'mAP@5':<10} {'mAP@10':<10} {'R@1':<10} {'R@5':<10} {'R@10':<10}")
    print("-"*80)
    
    for task_name in ['SBIR', 'TBIR', 'STBIR']:
        metrics = results['metrics'][task_name]
        print(f"{task_name:<10} "
              f"{metrics['mAP@1']:<10.4f} "
              f"{metrics['mAP@5']:<10.4f} "
              f"{metrics['mAP@10']:<10.4f} "
              f"{metrics['R@1']:<10.4f} "
              f"{metrics['R@5']:<10.4f} "
              f"{metrics['R@10']:<10.4f}")

save_results_summary()


✓ Results saved to: abc_evaluation_results.json

RESULTS TABLE (for paper)

Method     mAP@1      mAP@5      mAP@10     R@1        R@5        R@10      
--------------------------------------------------------------------------------
SBIR       0.4642     0.5258     0.5325     0.4642     0.6289     0.6795    
TBIR       0.0480     0.0764     0.0845     0.0480     0.1315     0.1933    
STBIR      0.4618     0.5381     0.5462     0.4618     0.6635     0.7232    
